# Cohort Data Creation

In [ ]:
import pandas as pd
from LabData.DataLoaders.GutMBLoader import GutMBLoader
from LabData.DataLoaders.SubjectLoader import SubjectLoader
from LabData.DataLoaders.DietLoggingLoader import DietLoggingLoader
from LabData.DataLoaders.LifeStyleLoader import LifeStyleLoader
from LabData.DataLoaders.BodyMeasuresLoader import BodyMeasuresLoader
from LabData.DataAnalyses.TenK_Trajectories.utils import get_diet_logging_around_stage
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


In [ ]:
home_path = '/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/'
SPECIES = 'segal_species' # 'segal_species' or 'mpa_species'

# Study configuration
study_ids = [15]  # [15] for Australian cohort, ['AUS'] for AUS
# Map study_ids to study name for file naming
study_name_map = {15: 'AU15', 'AUS': 'AUS'}
study_name = study_name_map.get(study_ids[0] if isinstance(study_ids[0], int) else study_ids[0], f'study_{study_ids[0]}')

min_col_present_frac = 0.05

In [ ]:
diet_mb_10k = pd.read_pickle(home_path + f"data/{SPECIES}/diet_mb.pkl")
with open(home_path + f'data/{SPECIES}/my_lists.pkl', 'rb') as file:
    loaded_lists = pickle.load(file)
base_features_10k, all_features_10k, targets_10k = loaded_lists
with open(home_path + f'data/{SPECIES}/scaler.pkl', 'rb') as scaler_file:
        scaler_10k = pickle.load(scaler_file)
diet_mb_10k

In [ ]:
def explore_columns(df):
    for column in df.columns:
        print(column)
        print(df[column].value_counts())

# study_ids is defined in cell 2
subjects_dl = SubjectLoader()
subjects_data = subjects_dl.get_data(groupby_reg='first', study_ids=study_ids)
subjects_df = subjects_data.df
print(subjects_df)

## Load Microbiome Data

In [ ]:
gut_bacteria = GutMBLoader().get_data(SPECIES, subjects_df=subjects_df, study_ids=study_ids,
                                groupby_reg='first', 
                                genotek_vals=[1], min_col_val=1e-4, take_log=True)
gut_bacteria_df = gut_bacteria.df.dropna(axis=1, how='all')
gut_bacteria_df.columns = gut_bacteria_df.columns.str.replace('s__', '')
# with open(home_path + f'data/{species}/my_lists.pkl', 'rb') as file:
#         loaded_lists = pickle.load(file)
# base_features, all_diet_features, targets = loaded_lists
gut_bacteria_df = gut_bacteria_df[targets_10k]



gut_bacteria_df.head(3)

In [ ]:
gut_bacteria_df.shape

In [ ]:
gut_bacteria_df

In [ ]:
gut_bacteria.df_metadata

In [ ]:
gut_bacteria_df = gut_bacteria_df.join(gut_bacteria.df_metadata[['RegistrationCode', 'Date']]).set_index(['RegistrationCode', 'Date'])
gut_bacteria_df.tail(20)

In [ ]:
gut_bacteria_df_date = gut_bacteria_df.copy()

In [ ]:
# Keep only baseline data (first test per person)
gut_bacteria_df = gut_bacteria_df.groupby(level=0).first().sort_index()

gut_bacteria_df

In [ ]:
# Normalize by row.

# Step 1: Convert to normal scale
gut_bacteria_df_normal = 10 ** gut_bacteria_df

# Step 2: Mask of values that are NOT 0.0001
mask = gut_bacteria_df_normal != 0.0001

# Step 3: Row-wise sum of the non-0.0001 values
non_floor_sum = gut_bacteria_df_normal.where(mask).sum(axis=1)

# Step 4: Normalize ONLY the non-0.0001 values, keep 0.0001 unchanged
gut_bacteria_df_normal = gut_bacteria_df_normal.where(~mask, gut_bacteria_df_normal.div(non_floor_sum, axis=0))

# Step 5: Convert back to log10
gut_bacteria_df_log = np.log10(gut_bacteria_df_normal)
gut_bacteria_df = gut_bacteria_df_log
gut_bacteria_df

In [ ]:
# Keep only baseline microbiome data
baseline_mb = gut_bacteria_df#.reset_index(level=[1], drop=True)

In [ ]:
gut_bacteria_df_col = gut_bacteria.df_columns_metadata
# View 5 most common bacteria
gut_bacteria_df_col[gut_bacteria_df_col['Unnamed: 0'].isin(["Rep_485", "Rep_609", "Rep_477", "Rep_449", "Rep_231"])]

In [ ]:
gut_bacteria_df_col.to_pickle(home_path + f"data/mb_names_{study_name.lower()}.pkl")

In [ ]:
gut_bacteria_df_meta = gut_bacteria.df_metadata
print(gut_bacteria_df_meta.head())
# Check that there's only one mb test per person
gut_bacteria_df_meta.RegistrationCode.value_counts()

### Alpha diversity targets

In [ ]:
# Richness
def richness(row):
    filtered = row[row > -4]
    return len(filtered)

baseline_mb['Richness'] = baseline_mb.apply(richness, axis=1)
baseline_mb

In [ ]:
# Shannon Diversity
def shannon(row):
    filtered = row[row > -4]
    filtered = filtered.drop("Richness")
    
    rel_abundance = 10 ** filtered
    ln_rel_abundance = np.log(rel_abundance.replace(0, 1))
    product = rel_abundance * ln_rel_abundance
    ans = -1 * product.sum()
    return round(float(ans), 2)

baseline_mb['Shannon_diversity'] = baseline_mb.apply(shannon, axis=1)
baseline_mb

In [ ]:
diversity_targets = ['Richness', 'Shannon_diversity']

## Load Diet Data

In [ ]:
with open('/net/mraid20/export/genie/LabData/Analyses/tomerse/david_colab/my_lists_diet.pkl', 'rb') as file:
            loaded_lists = pickle.load(file)
base_features, all_diet_features = loaded_lists

In [ ]:
with open('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/food_shortnames.pkl', 'rb') as file:
    food_shortnames = pickle.load(file)
food_shortnames

In [ ]:
# dll = DietLoggingLoader()
# dlld = dll.get_data(study_ids=15, stage='baseline')
# # log = get_diet_logging_around_stage(dlld.df, delta_before=2, delta_after=14)
# log = dlld.df.reset_index()
# log = log.set_index(['RegistrationCode','Date','food_id'])
# print(log.head(10))
# log.shape

In [ ]:
# log_date = dll.add_new_nutrients(log)
# log_date

In [ ]:
diet_aus = pd.read_csv('/net/mraid20/ifs/wisdom/segal_lab/genie/LabData/Analyses/nastya/diet_predictions/nutr_full_clean_aus.csv', index_col=0)
diet_aus

In [ ]:
nutr_list = diet_aus.columns
nutr_list

In [ ]:
# Unite similar features by summing them up
diet_aus['omega_6'] = diet_aus[['PUFA 18:2 n-6 c,c', 'PUFA 20:4 n-6', 'PUFA 18:2 CLAs', 'PUFA 18:2 i', 'PUFA 20:2 n-6 c,c', 'PUFA 2:4 n-6', 'PUFA 22:4', 'PUFA 18:3 n-6 c,c,c']].sum(axis=1)
diet_aus['omega_3'] = diet_aus[['PUFA 18:3 n-3 c,c,c (ALA)', 'PUFA 18:4', 'PUFA 20:5 n-3 (EPA)', 'PUFA 22:6 n-3 (DHA)', 'PUFA 22:5 n-3 (DPA)', 'PUFA 20:3 n-3']].sum(axis=1)
diet_aus['vitamin_E'] = diet_aus[['Vitamin E (alpha-tocopherol)', 'Tocopherol, beta', 'Tocopherol, delta', 'Tocopherol, gamma', 'Tocotrienol, alpha', 'Tocotrienol, beta', 'Tocotrienol, delta', 'Tocotrienol, gamma']].sum(axis=1)

# Drop the original columns after uniting
columns_to_drop = [
    'SFA 4:0', 'SFA 6:0', 'SFA 8:0', 'SFA 10:0', 'SFA 12:0', 'SFA 14:0', 'SFA 16:0', 'SFA 18:0', 'SFA 13:0', 'SFA 15:0', 'SFA 17:0', 'SFA 20:0', 'SFA 22:0', 'SFA 24:0',
    'MUFA 14:1', 'MUFA 15:1', 'MUFA 16:1', 'MUFA 17:1', 'MUFA 18:1', 'MUFA 18:1 c', 'MUFA 20:1', 'MUFA 22:1', 'MUFA 24:1 c', 'MUFA 16:1 c', 'MUFA 22:1 c', 'MUFA 18:1-11 t (18:1t n-7)',
    'PUFA 18:2 n-6 c,c', 'PUFA 20:4 n-6', 'PUFA 18:2 CLAs', 'PUFA 18:2 i', 'PUFA 20:2 n-6 c,c', 'PUFA 2:4 n-6', 'PUFA 22:4', 'PUFA 18:3 n-6 c,c,c',
    'PUFA 18:3 n-3 c,c,c (ALA)', 'PUFA 18:4', 'PUFA 20:5 n-3 (EPA)', 'PUFA 22:6 n-3 (DHA)', 'PUFA 22:5 n-3 (DPA)', 'PUFA 20:3 n-3',
    'TFA 16:1 t', 'TFA 18:1 t', 'TFA 18:2 t not further defined', 'TFA 18:2 t,t', 'TFA 22:1 t',
    'Vitamin A, IU', 'Carotene, beta', 'Retinol', 'Carotene, alpha', 'Cryptoxanthin, beta',
    'Vitamin D2 (ergocalciferol)', 'Vitamin D3 (cholecalciferol)', 'Vitamin D (D2 + D3), International Units', 
    'Vitamin K (Dihydrophylloquinone)', 'Vitamin K (Menaquinone-4)', 'Vitamin K (phylloquinone)',
    'Vitamin E (alpha-tocopherol)', 'Tocopherol, beta', 'Tocopherol, delta', 'Tocopherol, gamma', 'Tocotrienol, alpha', 'Tocotrienol, beta', 'Tocotrienol, delta', 'Tocotrienol, gamma', 'Vitamin E, added',
    'Folate, DFE', 'Sugars, total including NLEA',
    'Fatty acids, total trans-monoenoic', 'Fatty acids, total trans-polyenoic', 'Folate, DFE', 'Folate, food', 'Folic acid',
    'Galactose', 'Lactose', 'Maltose',  'PUFA 18:2', 'PUFA 18:3', 'PUFA 18:3i', 'PUFA 20:3', 'PUFA 20:4', 'PUFA 21:5',
    'Theobromine', 'Stigmasterol', 'Beta-sitosterol', 'Sucrose', 'matched_food_score',
    'Isoleucine', 'Leucine', 'Valine', 'Lysine', 'Threonine', 'Methionine', 'Phenylalanine', 'Tryptophan', 'Histidine',
    'Tyrosine', 'Arginine', 'Cystine', 'Serine', 'Alanine', 'Aspartic acid', 'Glutamic acid', 'Glycine', 'Hydroxyproline', 'Proline'
]

diet_aus.drop(columns=columns_to_drop, inplace=True)

# Optional: remove additional duplicates like 'Vitamin B-12, added' if needed
diet_aus.drop(columns=['Vitamin B-12, added'], inplace=True, errors='ignore')

# Ensure to keep only 'Folate, total'
diet_aus = diet_aus.loc[:, ~diet_aus.columns.duplicated()]


In [ ]:
# nutr_list = list(log_date.columns[4:])
# nutr_list

In [ ]:
with open('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/nutr_list_aus.pkl', 'wb') as file:
    pickle.dump(nutr_list, file)

### Filtering log date

In [ ]:
# ### Filter diet logging days to -2 up to 14 days before the stool collection date

# # Extract a Series mapping RegistrationCode -> stool_date
# stool_dates = gut_bacteria_df_date.index.get_level_values("Date")
# stool_dates = pd.Series(stool_dates.values,
#                         index=gut_bacteria_df_date.index.get_level_values("RegistrationCode"))

# # log_date index: (RegistrationCode, Date, food_id)
# log_rc = log_date.index.get_level_values("RegistrationCode")

# # Map RegistrationCode of each log entry to its stool date
# matched_stool = log_rc.map(stool_dates)

# log_dates = log_date.index.get_level_values("Date")

# lower_ok = log_dates >= (matched_stool - pd.Timedelta(days=2))
# upper_ok = log_dates <= (matched_stool + pd.Timedelta(days=14))

# mask = lower_ok & upper_ok

# filtered_log_date = log_date[mask]


In [ ]:
# filtered_log_date.shape

In [ ]:
# log_date = filtered_log_date.reset_index()
# log_date['Day'] = log_date['Date'].astype(str).str[:10]
# log_date['Hour'] = log_date['Date'].astype(str).str[10:16]
# log_date.drop('Date', axis=1, inplace=True)
# log_date = log_date.set_index(['RegistrationCode','Day','Hour','food_id'])
# print(log_date.head())

In [ ]:
# # Filter all entries with NaN Energy
# log_date = log_date[~log_date['Energy'].isna()]
# log_date.shape

In [ ]:
# len(set(log_date.index.get_level_values(0)))

In [ ]:
# # Filter foods that have no nutrient data, other than sugar substitutes
# # Filter rows where all nutrient values are 0
# log_date = log_date[~((log_date[nutr_list] == 0).all(axis=1))]
# log_date.shape

In [ ]:
# # Filter unrealistic energy values
# log_date = log_date[(log_date["Energy"] >= 0) & (log_date["Energy"] < 3000)]
# log_date.shape

In [ ]:
# # Filter unrealistic weight values
# log_date = log_date[(log_date["weight"] >= 0) & (log_date["weight"] < 2400)]
# log_date.shape

In [ ]:
# # Filter duplicate entries
# log_date_reset = log_date.reset_index()
# log_date = log_date_reset[
#     ~log_date_reset.duplicated(subset=['RegistrationCode', 'Day', 'food_id', 'Hour', 'weight'], keep='first')
# ].set_index(['RegistrationCode', 'Day', 'food_id', 'Hour'])
# log_date.shape

In [ ]:
# before_any_filters = log_date.reset_index()["RegistrationCode"].nunique()
# before_any_filters

In [ ]:
# # Filter logging days with <500 calories or >4000 calories

# total_energy_per_day = log_date.groupby(['RegistrationCode', 'Day'])['Energy'].transform('sum')
# log_date['total_energy_per_day'] = total_energy_per_day
# log_date = log_date[(log_date['total_energy_per_day'] >= 500) & (log_date['total_energy_per_day'] <= 4000)]
# log_date.shape

In [ ]:
# # Validate filtering
# log_date.groupby(['RegistrationCode', 'Day'])['Energy'].transform('sum').describe()

In [ ]:
# # Filter outlier days that might be under-documentation.

# # Assuming `log_date` is your DataFrame
# # Calculating energy per day
# log_date_energy_per_day = log_date.groupby(['RegistrationCode', 'Day'])['Energy'].sum().reset_index()

# # Grouping by RegistrationCode to calculate mean and std
# stats = log_date_energy_per_day.groupby('RegistrationCode')['Energy'].agg(['mean', 'std'])

# # Merging stats back to a new DataFrame
# log_date_with_stats = log_date_energy_per_day.merge(stats, on='RegistrationCode')

# # Defining outliers: Energy values outside mean ± 2*std
# log_date_with_stats['is_outlier_below'] = (log_date_with_stats['Energy'] < (log_date_with_stats['mean'] - 2.5 * log_date_with_stats['std'])) #| \
#                                     #(log_date_with_stats['Energy'] > (log_date_with_stats['mean'] + 2 * log_date_with_stats['std']))

# # Filtering outlier days
# # outliers = log_date_with_stats[log_date_with_stats['is_outlier']][['RegistrationCode', 'Day', 'Energy', 'is_outlier']]
# outliers = log_date_with_stats[log_date_with_stats['is_outlier_below']][['RegistrationCode', 'Day', 'Energy', 'is_outlier_below']]

# # Display the outliers
# print(outliers)
# # print(log_date_with_stats[log_date_with_stats['RegistrationCode'] == 'EXAMPLE_ID'])


In [ ]:
# # Filter outlier days
# log_date = log_date[~log_date_with_stats.set_index(['RegistrationCode', 'Day'])['is_outlier_below']]
# log_date.shape

In [ ]:
# log_date.head(20)

In [ ]:
# # Filter People with less than 3 days of diet documentation.
# # Group by RegistrationCode and count the number of unique days using the index level 'Day'
# unique_day_counts = log_date.groupby('RegistrationCode').apply(lambda x: x.index.get_level_values('Day').nunique())

# # Filter out RegistrationCodes with less than 3 unique days
# valid_registration_codes = unique_day_counts[unique_day_counts >= 3].index

# # Create a new DataFrame with only the valid RegistrationCodes
# log_date = log_date.loc[valid_registration_codes]

# # Display the filtered DataFrame
# print(log_date.shape)


In [ ]:
# people_more_than_3 = log_date.reset_index()["RegistrationCode"].nunique()
# people_more_than_3

In [ ]:
# people_more_than_8 = log_date.reset_index()["RegistrationCode"].nunique()
# people_more_than_8

In [ ]:
# before_any_filters

In [ ]:
# unique_day_counts

In [ ]:
# plt.hist(unique_day_counts, bins=range(1, unique_day_counts.max()+2))
# plt.xticks(range(1, unique_day_counts.max()+1))
# plt.show()


In [ ]:
# exact_7_days = (unique_day_counts == 7).sum()
# more_than_14_days = (unique_day_counts >= 14).sum()
# less_than_14_days = ((unique_day_counts < 14) & (unique_day_counts > 7)).sum()

# print(f"Number of unique_day_counts with exactly 7 days: {exact_7_days}")
# print(f"Number of unique_day_counts with 14 or more days: {more_than_14_days}")
# print(f"Number of unique_day_counts with less than 14 days: {less_than_14_days}")


In [ ]:
# days_saved = ((unique_day_counts < 8) & (unique_day_counts >= 3)).sum()

# print(f"Number of unique_day_counts with 4<=x<=7 days: {days_saved}")


In [ ]:
# # validate filtering:
# # Group by RegistrationCode and count the number of unique days using the index level 'Day'
# unique_day_counts_after = log_date.groupby('RegistrationCode').apply(lambda x: x.index.get_level_values('Day').nunique())
# plt.hist(unique_day_counts_after, bins=range(1, unique_day_counts_after.max()+2))
# plt.xticks(range(1, unique_day_counts_after.max()+1))
# plt.show()

In [ ]:
# Validate filtering:
# Group by RegistrationCode and count the number of unique days using the 'Date' index level
unique_day_counts_after = log_date.groupby('RegistrationCode').apply(
    lambda x: pd.to_datetime(x.index.get_level_values('Date')).normalize().nunique()
)

# Plot histogram of the unique day counts
plt.hist(unique_day_counts_after, bins=range(1, unique_day_counts_after.max() + 2))
plt.xticks(range(1, unique_day_counts_after.max() + 1))
plt.show()


In [ ]:
# log_date.groupby(['RegistrationCode', 'Day'])['Energy'].sum().describe()
# # max_calories_registration_code = log_date.groupby(['RegistrationCode', 'Day'])['Energy'].sum().idxmax()[0]
# # print(max_calories_registration_code)


In [ ]:
# log_date.reset_index()["RegistrationCode"].nunique()

In [ ]:
# log_date['total_energy_per_day']

In [ ]:
# nutr_list_no_energy = [nutrient for nutrient in nutr_list if nutrient != "Energy"]
# relative_nutrients = log_date[nutr_list_no_energy].div(log_date['total_energy_per_day'], axis=0)
# # relative_nutrients.to_pickle('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/relative_nutrients.pkl')

In [ ]:
# log_date[nutr_list_no_energy] = relative_nutrients

In [ ]:
# log_day = log_date.groupby(['RegistrationCode', 'Day'])[nutr_list].sum()
# log_day = log_day[log_day['Energy']>500]
# log_day = log_day[log_day['Energy']<4000]
# log_day

In [ ]:
# log_day['pct_protein_calories'] = (log_day['Protein']*4) * 100
# log_day['pct_carb_calories'] = (log_day['Carbohydrate, by difference']*4) * 100
# log_day['pct_fat_calories'] = (log_day['Total lipid (fat)']*9) * 100
# log_day['pct_alcohol_calories'] = (log_day['Alcohol, ethyl']*7) * 100
# log_day

In [ ]:
# log_day['sat_to_total_lipids_ratio'] = log_day['Fatty acids, total saturated'] / log_day['Total lipid (fat)']
# log_day['trans_to_total_lipids_ratio'] = log_day['Fatty acids, total trans'] / log_day['Total lipid (fat)']
# log_day['mono_to_total_lipids_ratio'] = log_day['Fatty acids, total monounsaturated'] / log_day['Total lipid (fat)']
# log_day['poly_to_total_lipids_ratio'] = log_day['Fatty acids, total polyunsaturated'] / log_day['Total lipid (fat)']
# log_day['omega3_to_total_lipids_ratio'] = log_day['omega_3'] / log_day['Total lipid (fat)']
# log_day['omega6_to_total_lipids_ratio'] = log_day['omega_6'] / log_day['Total lipid (fat)']

# # Compute and cap omega6_to_omega3_ratio at 100
# log_day['omega6_to_omega3_ratio'] = (
#     log_day['omega_6'] / log_day['omega_3'].replace(0, np.nan)
# ).replace([np.inf, -np.inf], np.nan).fillna(0).clip(upper=100)

# log_day['pct_saturated_fat_calories'] = (log_day['Fatty acids, total saturated']*9) * 100
# log_day['pct_saturated_fat_calories']


In [ ]:
# log_day = log_day.join(subjects_df[["age", "gender"]])
# log_day = log_day.reset_index(level=[1], drop=True)
# log_day = log_day.dropna()
# log_day

In [ ]:
# # Step 1: Reset index and drop "Day"
# log_reset = log_day.reset_index().drop("Date", axis=1)

# # Group by RegistrationCode and calculate mean for all numeric columns
# log_grouped = log_reset.groupby('RegistrationCode').mean()
# log_grouped.head()

In [ ]:
# diet_by_person = log_grouped.drop(columns=['Protein', 'Carbohydrate, by difference', 'Total lipid (fat)', 'Alcohol, ethyl', 'Fatty acids, total saturated', 'total_energy_per_day'])
# diet_by_person.head()

In [ ]:
# diet_by_person.isnull().sum().sort_values()

### END

In [ ]:
baseline_foods_all

In [ ]:
baseline_foods_all = baseline_foods_all.set_index('RegistrationCode')
baseline_foods = baseline_foods_all[[col for col in all_features_10k if col in baseline_foods_all.columns]]

In [ ]:
# Normalize by person to get mean % of daily calories
baseline_foods = baseline_foods.fillna(0)
baseline_foods = baseline_foods.div(baseline_foods.sum(axis=1), axis=0)
baseline_foods

In [ ]:
# baseline_nutrients = baseline_nutrients.set_index('RegistrationCode').drop(['Main score'], axis=1)
# baseline_nutrients

In [ ]:
# # Dictionary with the mappings for renaming
# rename_dict = {
#     'Fructose': 'Fructose',
#     'caffeine_mg': 'Caffeine',
#     'calcium_mg': 'Calcium, Ca',
#     'carbohydrate_g': 'Carbohydrate, by difference',
#     'cholesterol_mg': 'Cholesterol',
#     'iron_mg': 'Iron, Fe',
#     'magnesium_mg': 'Magnesium, Mg',
#     'niacin_mg': 'Niacin',
#     'phosphorus_mg': 'Phosphorus, P',
#     'potassium_mg': 'Potassium, K',
#     'protein_g': 'Protein',
#     'raevitamina_ug': 'Vitamin A, RAE',
#     'riboflavin_mg': 'Riboflavin',
#     'sodium_mg': 'Sodium, Na',
#     'thiamin_mg': 'Thiamin',
#     'totaldietaryfiber_g': 'Fiber, total dietary',
#     'totalfolate_ug': 'Folate, total',
#     'totallipid_g': 'Total lipid (fat)',
#     'totalmonounsaturatedfattyacids_g': 'Fatty acids, total monounsaturated',
#     'totalpolyunsaturatedfattyacids_g': 'Fatty acids, total polyunsaturated',
#     'totalsaturatedfattyacids_g': 'Fatty acids, total saturated',
#     'vitaminb12_ug': 'Vitamin B-12',
#     'vitaminb6_mg': 'Vitamin B-6',
#     'vitaminc_mg': 'Vitamin C, total ascorbic acid',
#     'vitamind_iu': 'Vitamin D (D2 + D3)',
#     'vitamine_mg': 'vitamin_E',
#     'zinc_mg': 'Zinc, Zn'
# }

# # Rename the items in the list using the mapping
# baseline_nutrients.columns = [rename_dict.get(item, item) for item in baseline_nutrients.columns]

# baseline_nutrients


In [ ]:
# food_cat_baseline = food_cat_baseline.set_index('RegistrationCode')
# # Normalize by person to get mean % of daily calories
# food_cat_baseline = food_cat_baseline.div(food_cat_baseline.sum(axis=1), axis=0)
# food_cat_baseline

## Combine Dataframes

In [ ]:
# baseline_nutrients = baseline_nutrients.set_index('RegistrationCode')
# baseline_nutrients.columns = [rename_dict.get(item, item) for item in baseline_nutrients.columns]
# baseline_nutrients

In [ ]:
# diet_mb = baseline_foods.join(baseline_nutrients, how='inner')
# # diet_mb = diet_mb.dropna()
# diet_mb

In [ ]:
subjects_df = subjects_df.reset_index(level=[1], drop=True)

In [ ]:
subjects_df

In [ ]:
base_features = ["age", "gender"]
subjects_df.index = subjects_df.index.astype('int')
baseline_foods = baseline_foods.join(subjects_df[base_features])
baseline_foods = baseline_foods.rename(columns={'gender': 'sex'})
# diet_mb = diet_mb.reset_index(level=[1], drop=True)
# diet_mb = diet_mb.dropna()
baseline_foods

In [ ]:
aus_diet_features = baseline_foods.columns
aus_diet_features

In [ ]:
aus_10k_shared_features = [col for col in aus_diet_features if col in all_features_10k]
aus_10k_shared_features.remove('Fructose')
aus_10k_shared_features

In [ ]:
len(aus_10k_shared_features)

In [ ]:
baseline_foods = baseline_foods[aus_10k_shared_features]

In [ ]:
baseline_mb.index = baseline_mb.index.astype('int')
diet_mb_baseline = baseline_foods.join(baseline_mb)
diet_mb_baseline

In [ ]:
# Baseline dataframe is already created in cell 42
diet_mb_baseline

In [ ]:
# Baseline data shape
print(diet_mb_baseline.shape)

In [ ]:
# from sklearn.preprocessing import StandardScaler

# with open(home_path + f'data/{SPECIES}/age_scaler.pkl', 'rb') as f:
#     age_scaler = pickle.load(f)

# with open(home_path + f'data/{SPECIES}/mb_scaler.pkl', 'rb') as f:
#     mb_scaler = pickle.load(f)

# with open(home_path + f'data/{SPECIES}/div_scaler.pkl', 'rb') as f:
#     div_scaler = pickle.load(f)

# diet_scaler = StandardScaler()
# diet_mb_10k_scaled = diet_scaler.fit_transform(diet_mb_10k[aus_10k_shared_features])

# # # Apply the scaler to the dataframe
# diet_mb_baseline.loc[:, aus_10k_shared_features] = diet_scaler.transform(diet_mb_baseline[aus_10k_shared_features])
# diet_mb_baseline.loc[:, ["age"]] = age_scaler.transform(diet_mb_baseline[["age"]])
# diet_mb_baseline.loc[:, targets_10k] = mb_scaler.transform(diet_mb_baseline[targets_10k])
# diet_mb_baseline.loc[:, ["Richness", "Shannon_diversity"]] = div_scaler.transform(diet_mb_baseline[["Richness", "Shannon_diversity"]])
# # diet_mb_baseline[aus_10k_shared_features] = pd.DataFrame(scaler.transform(diet_mb_baseline[aus_10k_shared_features]), columns=diet_mb_baseline[aus_10k_shared_features].columns, index=diet_mb_baseline[aus_10k_shared_features].index)
# diet_mb_baseline.describe()

In [ ]:
# Scaling code for baseline (commented out)
# diet_mb_baseline.loc[:, aus_10k_shared_features] = diet_scaler.transform(diet_mb_baseline[aus_10k_shared_features])
# diet_mb_baseline.loc[:, ["age"]] = age_scaler.transform(diet_mb_baseline[["age"]])
# diet_mb_baseline.loc[:, targets_10k] = mb_scaler.transform(diet_mb_baseline[targets_10k])
# diet_mb_baseline.loc[:, ["Richness", "Shannon_diversity"]] = div_scaler.transform(diet_mb_baseline[["Richness", "Shannon_diversity"]])
# diet_mb_baseline.describe()

In [ ]:
# Count rows that have at least one NaN
nan_rows = diet_mb_baseline[diet_mb_baseline.isna().any(axis=1)]
num_nan_rows = len(nan_rows)

# Find which columns contain NaN values
nan_features = diet_mb_baseline.columns[diet_mb_baseline.isna().any()].tolist()

print(f"Number of rows with at least one NaN: {num_nan_rows}")
print("Features that contain NaN values:")
print(nan_features)


In [ ]:
print("\nNaN count per column:")
print(diet_mb_baseline.isna().sum()[diet_mb_baseline.isna().sum() > 0])
diet_mb_baseline = diet_mb_baseline.dropna(how='any')


In [ ]:
print(diet_mb_baseline.shape)

In [ ]:
diet_mb_baseline.to_pickle(home_path + f'data/diet_mb_{study_name.lower()}_baseline.pkl')
with open(home_path + f'data/my_lists_{study_name.lower()}.pkl', 'wb') as file:
    pickle.dump([aus_10k_shared_features, targets_10k], file)

In [ ]:
print(len(aus_10k_shared_features))
print(len(targets_10k))